[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance4_correction.ipynb)

# Séance 3.4 — Régression linéaire — expliquer, et de combien

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'étude de cas en binôme)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- ajuster une régression avec `smf.ols("y ~ x", donnees).fit()`
- lire un coefficient, sa p-value et son intervalle de confiance
- dire ce que le R² mesure — et ce qu'il ne mesure pas
- interpréter un coefficient « toutes choses égales par ailleurs »
- faire entrer une variable qualitative dans un modèle
- reconnaître une extrapolation et refuser d'y répondre

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")   ## une ligne = une commande
cli = pd.read_csv(BASE + "clients_ca.csv")   ## une ligne = un client

print(cmd.shape, cli.shape)
cli.head(3)

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Votre première régression

> **Votre mission :**
> - Ajuster le modèle `ca ~ qte` sur `cmd` → `mq`.
> - Mettre le coefficient de `qte` dans `coef_qte`, arrondi à 3 décimales.
> - Ne pas oublier `.fit()`.

In [ ]:
mq = smf.ols("ca ~ qte", cmd).fit()   ## "ca expliquee par qte"

# params contient les coefficients, indexes par nom de variable
coef_qte = round(mq.params["qte"], 3)
print("un article de plus :", coef_qte, "euros")

In [ ]:
verifier("1 - coefficient de qte", coef_qte == 1.436,
         "la formule s'ecrit \"ca ~ qte\", et il faut .fit() a la fin")

### Exercice 2 — Ce coefficient est-il crédible ?

> **Votre mission :**
> - Récupérer la p-value du coefficient de `qte` → `p_qte`, arrondie à 4 décimales.
> - Conclut-on au seuil de 5 % ? → `retenu` (`True` ou `False`)

In [ ]:
# pvalues est construit comme params : un nombre par variable
p_qte = round(mq.pvalues["qte"], 4)
retenu = p_qte < 0.05   ## le meme seuil qu'en seance 3.2

print("p =", p_qte, "| coefficient retenu :", retenu)

In [ ]:
verifier("2a - p-value du coefficient", p_qte == 0.0, "mq.pvalues['qte']")
verifier("2b - conclusion", bool(retenu), "comparez la p-value au seuil de 0,05")

### Exercice 3 — Quelle part le modèle explique-t-il ?

> **Votre mission :**
> - Mettre le R² du modèle dans `r2_qte`, arrondi à 3 décimales.
> - Le traduire en phrase : quelle part de la variation des montants ce modèle reproduit-il ?

In [ ]:
r2_qte = round(mq.rsquared, 3)   ## un attribut, sans parentheses

print("le modele reproduit", round(100 * r2_qte), "% de la variation")

In [ ]:
verifier("2 - R2 du modele", r2_qte == 0.719,
         "l'attribut s'appelle rsquared, sans parentheses")

### Exercice 4 — L'intercept, et son piège

> **Votre mission :**
> - Mettre l'intercept dans `origine`, arrondi à 2 décimales.
> - C'est le montant prédit pour une commande de **zéro article**.
> - Mettre `True` dans `interpretable` si ce nombre a un sens commercial, `False` sinon.

In [ ]:
origine = round(mq.params["Intercept"], 2)   ## avec une majuscule

# Une commande de zero article a 114 EUR n'existe pas : l'intercept
# sert a caler la droite, il ne s'interprete pas toujours
interpretable = False

print(origine, "euros pour zero article | interpretable :", interpretable)

In [ ]:
verifier("3a - intercept", origine == 114.59, "il s'appelle Intercept, avec une majuscule")
verifier("3b - interpretation", interpretable is False,
         "une commande de zero article, ca n'existe pas")

### Exercice 5 — La fourchette du coefficient

> **Votre mission :**
> - Un coefficient s'annonce toujours avec son intervalle de confiance.
> - Récupérer les deux bornes de celui de `qte` → `bas_qte` et `haut_qte`, arrondies à 3 décimales.
> - *Nouveau :* `mq.conf_int()` renvoie un tableau à deux colonnes, une ligne par variable.

In [ ]:
bornes = mq.conf_int().loc["qte"]   ## deux colonnes : bas, haut

bas_qte = round(bornes[0], 3)    ## borne basse
haut_qte = round(bornes[1], 3)   ## borne haute

# On annonce "environ 1,44 EUR par article, entre 1,40 et 1,48",
# jamais "1,4363 EUR"
print("entre", bas_qte, "et", haut_qte, "euros par article")

In [ ]:
verifier("4a - borne basse", bas_qte == 1.397, "conf_int().loc['qte'] puis la position 0")
verifier("4b - borne haute", haut_qte == 1.476, "la seconde colonne de conf_int()")

### Exercice 6 — Ajouter une variable

> **Votre mission :**
> - Ajuster `ca ~ qte + nart` → `mq2`, puis relever le nouveau coefficient de `qte` → `coef_qte2` et le R² → `r2_qte2`.
> - Le coefficient de `qte` bouge-t-il beaucoup ? Et le R² ?

In [ ]:
# Les variables explicatives se separent par un +
mq2 = smf.ols("ca ~ qte + nart", cmd).fit()

coef_qte2 = round(mq2.params["qte"], 3)   ## a nombre de produits distincts egal
r2_qte2 = round(mq2.rsquared, 3)          ## deux milliemes de mieux
print(coef_qte2, "contre", coef_qte, "| R2 :", r2_qte2, "contre", r2_qte)

# 1,403 contre 1,436, et le R2 gagne deux milliemes : ajouter nart
# n'apporte presque rien QUAND qte est deja la. Comparez avec le cours,
# ou l'ajout de qte divisait le coefficient de nart par huit.

In [ ]:
verifier("5a - coefficient de qte", coef_qte2 == 1.403, "la formule est \"ca ~ qte + nart\"")
verifier("5b - R2 du modele complet", r2_qte2 == 0.721, "rsquared du nouveau modele")

### Exercice 7 — Une variable qualitative

> **Votre mission :**
> - Ajuster `ca ~ qte + jour` → `mj`, puis afficher coefficients et p-values.
> - Mettre le nombre de jours **significatifs** au seuil de 5 % dans `nb_jours_sig`.
> - Rappel : une modalité sert de référence et n'apparaît pas dans le tableau.

In [ ]:
mj = smf.ols("ca ~ qte + jour", cmd).fit()   ## jour est du texte

resume = pd.DataFrame({"coef": mj.params.round(2), "p": mj.pvalues.round(3)})
print(resume)

# filter(like="jour") ne garde que les lignes dont l'etiquette
# contient "jour" : les modalites, sans l'intercept ni qte
jours = resume.filter(like="jour", axis=0)
nb_jours_sig = (jours["p"] < 0.05).sum()
print(nb_jours_sig, "jour(s) significatif(s)")

# Aucun. A nombre d'unites egal, le jour de la semaine ne change
# rien au montant — c'est exactement la conclusion de la seance 3.2,
# retrouvee par un autre chemin.

In [ ]:
verifier("6 - jours significatifs", nb_jours_sig == 0,
         "comparez chaque p-value de modalite au seuil de 0,05")

### Exercice 8 — Là où le modèle se trompe le plus

> **Votre mission :**
> - Un **résidu** est l'écart entre la valeur observée et la valeur prédite.
> - Mettre le plus grand résidu de `mq` dans `res_max` (arrondi à 2 décimales), et afficher la commande correspondante.
> - *Nouveau :* `mq.resid` est la série des résidus, alignée sur les lignes de `cmd`.

In [ ]:
res_max = round(mq.resid.max(), 2)   ## observe moins predit
ligne = cmd.loc[mq.resid.idxmax()]   ## idxmax() : QUELLE commande

print("sous-estimation de", res_max, "euros")
print(ligne[["cmd_id", "ca", "qte", "pays"]])

# La plus grosse commande du fichier (16 775 EUR, Irlande) est
# sous-estimee de 7 757 EUR : ses produits valent bien plus cher que la
# moyenne. Les residus disent OU un modele echoue, et c'est souvent la
# que se trouve l'information interessante.

In [ ]:
verifier("7 - plus grand residu", res_max == 7756.88,
         "resid est une serie : max() donne la valeur, idxmax() la ligne")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Le contexte

> **Votre mission :**
> Le comité de direction veut **augmenter le panier moyen** et vous demande une note d'une page : *sur quoi faut-il agir, et de combien peut-on espérer bouger ?* Les sept étapes ci-dessous vous y mènent — la dernière est le livrable. Travaillez en binôme, une tablette pour le code, une pour les notes.

In [ ]:
print(len(cmd), "commandes |", len(cli), "clients")

In [ ]:
verifier("0 - donnees pretes", len(cmd) == 1955 and len(cli) == 472,
         "relancez la cellule de preparation")

### Étape 1 — De quoi parle-t-on ? (rappel 3.1)

> **Votre mission :**
> - Sur `cli` (un client par ligne), calculer la moyenne et la médiane du `ca` → `moy_cli` et `med_cli`, arrondies à 2 décimales.
> - Lequel des deux mettriez-vous dans la note ?

In [ ]:
moy_cli = round(cli["ca"].mean(), 2)     ## 2 443 EUR
med_cli = round(cli["ca"].median(), 2)   ## 805 EUR : le client typique

print("moyenne", moy_cli, "| mediane", med_cli)

# 2 443 EUR contre 805 EUR : un facteur trois. Le client typique
# depense 805 EUR ; la moyenne est tiree par une poignee de grossistes.

In [ ]:
verifier("1a - CA moyen par client", moy_cli == 2442.61, "mean()")
verifier("1b - CA median par client", med_cli == 805.3, "median()")

### Étape 2 — De combien ? La première régression

> **Votre mission :**
> - Ajuster `ca ~ ncmd` sur `cli` → `mod1`.
> - Récupérer le coefficient de `ncmd` → `coef_ncmd` (2 décimales) et le R² → `r2_1` (3 décimales).

In [ ]:
# Ne pas oublier .fit() : sans lui, le modele n'est pas ajuste
mod1 = smf.ols("ca ~ ncmd", cli).fit()

coef_ncmd = round(mod1.params["ncmd"], 2)   ## par commande supplementaire
r2_1 = round(mod1.rsquared, 3)              ## 75 % de la variation
print("chaque commande supplementaire :", coef_ncmd, "euros | R2 :", r2_1)

In [ ]:
verifier("2a - coefficient de ncmd", coef_ncmd == 759.74,
         "la formule s'ecrit \"ca ~ ncmd\", et il faut .fit()")
verifier("2b - R2 du modele", r2_1 == 0.754, "mod1.rsquared")

### Étape 3 — Une variable qui n'apporte rien

> **Votre mission :**
> - L'ancienneté du client (`anc`, en jours depuis l'inscription) devrait compter. Vérifions.
> - Ajuster `ca ~ ncmd + anc` → `mod2`. Récupérer la p-value de `anc` → `p_anc` (3 décimales) et le R² → `r2_2`.
> - Comparez `r2_2` à `r2_1`.

In [ ]:
mod2 = smf.ols("ca ~ ncmd + anc", cli).fit()

p_anc = round(mod2.pvalues["anc"], 3)   ## 0,241 : rien a retenir
r2_2 = round(mod2.rsquared, 3)          ## identique a r2_1
print("p-value de l'anciennete :", p_anc, "| R2 :", r2_2, "contre", r2_1)

# p = 0,241 : rien ne permet de dire que l'anciennete joue. Et le R2 ne
# bouge pas d'un millieme. Une variable de plus n'est pas une information
# de plus : ici, l'anciennete n'explique rien que ncmd n'explique deja.

In [ ]:
verifier("3a - p-value de l'anciennete", p_anc == 0.241, "mod2.pvalues['anc']")
verifier("3b - R2 inchange", r2_2 == 0.754,
         "comparez avec r2_1 : la variable ajoutee n'apporte rien")

### Étape 4 — Le coefficient qui change de sens

> **Votre mission :**
> - On revient aux commandes. Ajuster `ca ~ nart` puis `ca ~ nart + qte` sur `cmd`.
> - Relever le coefficient de `nart` dans chacun → `c_seul` et `c_avec_qte` (2 décimales).
> - Comment expliqueriez-vous cet écart au comité ?

In [ ]:
ma = smf.ols("ca ~ nart", cmd).fit()         ## nart porte tout le merite
mb = smf.ols("ca ~ nart + qte", cmd).fit()   ## a nombre d'unites egal

c_seul = round(ma.params["nart"], 2)         ## 15,93
c_avec_qte = round(mb.params["nart"], 2)     ## 2,05
print("nart seul :", c_seul, "| nart avec qte :", c_avec_qte)

# 15,93 puis 2,05. Dans le premier modele, nart portait aussi l'effet
# des quantites. Le second le lit "a nombre d'unites egal". Les deux
# sont justes : ils ne repondent pas a la meme question.

In [ ]:
verifier("4a - nart seul", c_seul == 15.93, "modele a une seule variable")
verifier("4b - nart avec qte", c_avec_qte == 2.05,
         "ajoutez qte a la formule, separe par un +")

### Étape 5 — Les marchés, à quantité égale

> **Votre mission :**
> - Sur les quatre pays les plus présents, ajuster `ca ~ qte + pays` → `mod4`.
> - Relever le coefficient du Royaume-Uni → `coef_uk` (2 décimales) et la p-value de l'Irlande → `p_irl` (3 décimales).
> - Souvenez-vous de la séance 3.2 : le panier irlandais y était bien plus élevé.

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

# pays est du texte : la regression le decoupe en comparaisons a une
# modalite de reference (ici l'Allemagne, premiere par ordre alphabetique)
mod4 = smf.ols("ca ~ qte + pays", sub).fit()
coef_uk = round(mod4.params["pays[T.Royaume-Uni]"], 2)   ## ecart a l'Allemagne
p_irl = round(mod4.pvalues["pays[T.Irlande]"], 3)        ## 0,346 : plus rien
print("Royaume-Uni :", coef_uk, "euros | p Irlande :", p_irl)

# L'ecart irlandais de la seance 3.2 disparait (p = 0,346) une fois les
# quantites prises en compte : les commandes irlandaises ne sont pas plus
# cheres par article, elles sont plus GROSSES.

In [ ]:
verifier("5a - coefficient du Royaume-Uni", coef_uk == -99.59,
         "la formule est \"ca ~ qte + pays\"")
verifier("5b - p-value de l'Irlande", p_irl == 0.346,
         "mod4.pvalues, avec l'etiquette pays[T.Irlande]")

### Étape 6 — Prédire, et savoir s'arrêter

> **Votre mission :**
> - Avec `ma` (le modèle `ca ~ nart`), prédire le montant d'une commande à 30 produits distincts → `pred30` (2 décimales).
> - Puis d'une commande à 1000 produits distincts → `pred1000`.
> - Laquelle des deux prédictions refuseriez-vous de communiquer, et pourquoi ?

In [ ]:
pred30 = round(ma.predict(pd.DataFrame({"nart": [30]})).iloc[0], 2)     ## observe
pred1000 = round(ma.predict(pd.DataFrame({"nart": [1000]})).iloc[0], 2) ## jamais vu

print(pred30, "euros |", pred1000, "euros")
print("maximum observe :", cmd["nart"].max(), "produits distincts")

# 30 produits distincts : dans le domaine observe, la prediction se defend.
# 1000 produits distincts : le modele n'a jamais rien vu au-dela de 259. Il
# repond quand meme, sans prevenir. C'est une extrapolation.

In [ ]:
verifier("6a - prediction a 30 produits distincts", pred30 == 699.98,
         "predict() attend un DataFrame avec la meme colonne")
verifier("6b - prediction a 1000 produits distincts", pred1000 == 16156.37,
         "meme commande, avec 1000 a la place de 30")

### Étape 7 — Le livrable

> **Votre mission :**
> - Vous avez tout ce qu'il faut. Rédigez la note en commentaire, dans la cellule ci-dessous.
> - **Format imposé :** trois constats chiffrés, puis une recommandation, puis une limite que vous assumez.
> - Contrainte : chaque chiffre annoncé doit venir d'une étape précédente, et être accompagné de ce qui le rend crédible (p-value, effectif ou intervalle).

In [ ]:
print("CA median par client   :", med_cli)
print("Effet d'une commande   :", coef_ncmd, "euros (R2", r2_1, ")")
print("Effet d'un produit distinct :", c_avec_qte, "euros a quantite egale")
print("Ecart Royaume-Uni      :", coef_uk, "euros a quantite egale")

# NOTE AU COMITE — une redaction possible
#
# Constat 1. Le client typique depense 805 EUR sur la periode. La moyenne
#   de 2 443 EUR ne decrit personne : elle est tiree par une poignee de
#   grossistes.
# Constat 2. Le levier principal est la FREQUENCE, pas la taille du panier :
#   une commande supplementaire par client vaut en moyenne 760 EUR, et le
#   nombre de commandes reproduit a lui seul 75 % de la variation du CA.
# Constat 3. Pousser un produit distinct de plus dans un panier ne rapporte que
#   2,05 EUR a nombre d'unites egal — huit fois moins que ce que
#   suggere une lecture rapide (15,93 EUR).
# Recommandation. Concentrer l'effort commercial sur le reachat (frequence)
#   plutot que sur l'elargissement du panier en caisse.
# Limite. Ces chiffres decrivent des ASSOCIATIONS sur douze mois de
#   donnees, pas des effets d'une action. Seul un test A/B permettrait
#   d'affirmer qu'une relance produit reellement une commande de plus.

In [ ]:
verifier("7 - resultats disponibles pour la note",
         med_cli == 805.3 and coef_ncmd == 759.74 and c_avec_qte == 2.05,
         "reprenez les etapes 1, 2 et 4 avant de rediger")